In [ ]:
!aws s3 cp --no-sign-request \
s3://noaa-nws-graphcastgfs-pds/graphcastgfs.20250730/00/forecasts_13_levels/graphcastgfs.t00z.pgrb2.0p25.f006 \
.


In [2]:
import xarray as xr

In [3]:
ds = xr.open_dataset(
    'graphcastgfs.t00z.pgrb2.0p25.f006',
    engine='cfgrib',
    backend_kwargs={'filter_by_keys': {'centre': 'kwbc'}}
)
print(ds)


<xarray.Dataset>
Dimensions:     (latitude: 721, longitude: 1440)
Coordinates:
    time        datetime64[ns] ...
    step        timedelta64[ns] ...
    meanSea     float64 ...
  * latitude    (latitude) float64 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    valid_time  datetime64[ns] ...
    surface     float64 ...
Data variables:
    prmsl       (latitude, longitude) float32 ...
    tp          (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2025-07-31T21:55 GRIB to CDM+CF via cfgrib-0.9.1...


In [7]:
# Extract approximate Bhutan region
# consistent with how I crop ERA5 data

bhutan_ds = ds.sel(
    latitude=slice(28.5, 26.5),
    longitude=slice(88.5, 92)
)

print(bhutan_ds)

<xarray.Dataset>
Dimensions:     (latitude: 9, longitude: 15)
Coordinates:
    time        datetime64[ns] ...
    step        timedelta64[ns] ...
    meanSea     float64 ...
  * latitude    (latitude) float64 28.5 28.25 28.0 27.75 ... 27.0 26.75 26.5
  * longitude   (longitude) float64 88.5 88.75 89.0 89.25 ... 91.5 91.75 92.0
    valid_time  datetime64[ns] ...
    surface     float64 ...
Data variables:
    prmsl       (latitude, longitude) float32 ...
    tp          (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2025-07-31T21:46 GRIB to CDM+CF via cfgrib-0.9.1...


In [5]:
import cfgrib

# Load all message groups from the GRIB2 file
datasets = cfgrib.open_datasets(
    'graphcastgfs.t00z.pgrb2.0p25.f006',
    filter_by_keys={'centre': 'kwbc'}
)

# Check what’s inside
print(f"Found {len(datasets)} datasets")
for i, ds in enumerate(datasets):
    print(f"\n--- Dataset {i} ---")
    print("Variables:", list(ds.data_vars))
    print("Dimensions:", ds.dims)


Found 5 datasets

--- Dataset 0 ---
Variables: ['u10', 'v10']
Dimensions: Frozen({'latitude': 721, 'longitude': 1440})

--- Dataset 1 ---
Variables: ['t2m']
Dimensions: Frozen({'latitude': 721, 'longitude': 1440})

--- Dataset 2 ---
Variables: ['t', 'u', 'v', 'q', 'w', 'gh']
Dimensions: Frozen({'isobaricInhPa': 13, 'latitude': 721, 'longitude': 1440})

--- Dataset 3 ---
Variables: ['prmsl']
Dimensions: Frozen({'latitude': 721, 'longitude': 1440})

--- Dataset 4 ---
Variables: ['tp']
Dimensions: Frozen({'latitude': 721, 'longitude': 1440})
